In [44]:

import kagglehub
import pandas as pd
import numpy as np
import os

In [45]:
path = kagglehub.dataset_download("xfkzujqjvx97n/football-datasets")
tables = {}
print("Path to dataset files:", path)
for root, dirs, files in os.walk(path):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), path))
        if f.endswith(".csv"):
            nom = os.path.splitext(f)[0]
            tables[nom] = pd.read_csv(os.path.join(root, f), low_memory=False)


Path to dataset files: /Users/leonardpatron/.cache/kagglehub/datasets/xfkzujqjvx97n/football-datasets/versions/2
player_profiles/player_profiles.csv
player_national_performances/player_national_performances.csv
player_performances/player_performances.csv
player_injuries/player_injuries.csv
team_children/team_children.csv
team_competitions_seasons/team_competitions_seasons.csv
player_latest_market_value/player_latest_market_value.csv
player_market_value/player_market_value.csv
player_teammates_played_with/player_teammates_played_with.csv
team_details/team_details.csv
transfer_history/transfer_history.csv


In [46]:
for nom, df in tables.items():
    print(f"\n=== {nom} ===")
    print(df.shape)
    print(df.columns.tolist())


=== player_profiles ===
(92671, 34)
['player_id', 'player_slug', 'player_name', 'player_image_url', 'name_in_home_country', 'date_of_birth', 'place_of_birth', 'country_of_birth', 'height', 'citizenship', 'is_eu', 'position', 'main_position', 'foot', 'current_club_id', 'current_club_name', 'joined', 'contract_expires', 'outfitter', 'social_media_url', 'player_agent_id', 'player_agent_name', 'contract_option', 'date_of_last_contract_extension', 'on_loan_from_club_id', 'on_loan_from_club_name', 'contract_there_expires', 'second_club_url', 'second_club_name', 'third_club_url', 'third_club_name', 'fourth_club_url', 'fourth_club_name', 'date_of_death']

=== player_national_performances ===
(92701, 9)
['player_id', 'team_id', 'matches', 'goals', 'shirt_number', 'debut', 'coach_id', 'debut_game_id', 'career_state']

=== player_performances ===
(1878719, 20)
['player_id', 'season_name', 'competition_id', 'competition_name', 'team_id', 'team_name', 'nb_in_group', 'nb_on_pitch', 'goals', 'assist

In [47]:
# Test de l'unicité des clés primaires
for t, k in [("player_profiles", "player_id"), ("team_details", "club_id"),
             ("player_latest_market_value", "player_id")]:
    print(f"{t}.{k} unique :", tables[t][k].is_unique)

player_profiles.player_id unique : True
team_details.club_id unique : True
player_latest_market_value.player_id unique : True


In [48]:
def check_lien(src, col, ref, ref_col):
    s, r = tables[src][col].dropna(), tables[ref][ref_col].dropna()
    return {"lien": f"{src}.{col} → {ref}.{ref_col}",
            "% distincts": round(pd.Series(s.unique()).isin(r).mean() * 100, 1),
            "% lignes": round(s.isin(r).mean() * 100, 1)}
pd.set_option("display.max_colwidth", None)
liens = [
    # Couverture : part des joueurs de player_profiles présents dans chaque table
    ("player_profiles", "player_id", "player_performances", "player_id"),
    ("player_profiles", "player_id", "transfer_history", "player_id"),
    ("player_profiles", "player_id", "player_market_value", "player_id"),
    ("player_profiles", "player_id", "player_latest_market_value", "player_id"),
    ("player_profiles", "player_id", "player_injuries", "player_id"),
    ("player_injuries", "player_id", "player_profiles", "player_id"),
    ("player_profiles", "player_id", "player_national_performances", "player_id"),
    ("player_profiles", "player_id", "player_teammates_played_with", "player_id"),
    ("player_profiles", "player_id", "player_teammates_played_with", "teammate_player_id"),
    # Liens vers les clubs 
    ("player_performances", "team_id", "team_details", "club_id"),
    ("transfer_history", "from_team_id", "team_details", "club_id"),
    ("transfer_history", "to_team_id", "team_details", "club_id"),
    ("player_profiles", "current_club_id", "team_details", "club_id"),
    ("team_children", "parent_team_id", "team_details", "club_id"),
    ("team_children", "child_team_id", "team_details", "club_id"),
    ("player_national_performances", "team_id", "team_details", "club_id"),
    ("team_competitions_seasons", "club_id", "team_details", "club_id"),
    # Liens vers les compétitions (sens inchangé)
    ("player_performances", "competition_id", "team_competitions_seasons", "competition_id"),
    ("team_details", "competition_id", "team_competitions_seasons", "competition_id"),
]
display(pd.DataFrame([check_lien(*l) for l in liens]))

,lien,% distincts,% lignes
0,player_profiles.player_id → player_performances.player_id,95.4,95.4
1,player_profiles.player_id → transfer_history.player_id,98.4,98.4
2,player_profiles.player_id → player_market_value.player_id,74.9,74.9
3,player_profiles.player_id → player_latest_market_value.player_id,74.9,74.9
4,player_profiles.player_id → player_injuries.player_id,37.3,37.3
5,player_injuries.player_id → player_profiles.player_id,100.0,100.0
6,player_profiles.player_id → player_national_performances.player_id,30.4,30.4
7,player_profiles.player_id → player_teammates_played_with.player_id,91.1,91.1
8,player_profiles.player_id → player_teammates_played_with.teammate_player_id,56.9,56.9
9,player_performances.team_id → team_details.club_id,16.0,76.3


**Le noyeau des données de base dont on aura besoin plus tard est solide.** Les quatres tables indispensables (`player_profiles`, `player_performances`, `player_injuries`, `player_market_value`) sont reliées par player_id. Toutes les lignes des tables du noyau correspondent à un joueur connu, mais la couverture des joueurs varie : 95 % ont des performances, 75 % une valeur marchande et 37 % au moins une blessure.

**Les faiblesses portent sur les clubs.** Ce sont plus des variables de contexte.

Les liens entre les tables `team_details` et `team_competitions_seasons` est établie à 100% dans les deux sens. Cependant les liens depuis les autres tables sont plus faibles.

player_performances (76%) :  table clé pour attribuer à chaque (joueur, saison) un club. À voir si les 24% manquants concernent de petits club.

transfer_history (48 ~ 50 %) : lien faible mais pas bloquant car le modèle de perte devrait reposer sur d’autres variables que les équipes.

player_profiles, team_children et player_national_performances : inutile ou usage secondaire pour les modélisation.

player_profiles.current_team_id : inutile pour une base historique 



In [49]:
joueurs = tables["player_profiles"]["player_id"]

tables_infos_joueurs = {
    "player_performances": "player_id",
    "player_market_value": "player_id",
    "player_latest_market_value": "player_id",
    "player_injuries": "player_id",
    "transfer_history": "player_id",
    "player_national_performances": "player_id",
    "player_teammates_played_with": "player_id",
}

couverture = pd.DataFrame({
    "table": list(tables_infos_joueurs),
    "% joueurs couverts": [round(joueurs.isin(tables[t][c]).mean() * 100, 1)
                           for t, c in tables_infos_joueurs.items()],
}).sort_values("% joueurs couverts", ascending=False)
display(couverture)

# Couverture combinée des infos indispensables au sujet
a_perf  = joueurs.isin(tables["player_performances"]["player_id"])
a_value = joueurs.isin(tables["player_market_value"]["player_id"])
a_inj   = joueurs.isin(tables["player_injuries"]["player_id"])

print("Performances + valeur marchande :", round((a_perf & a_value).mean() * 100, 1), "%")
print("Performances + valeur + au moins une blessure :", round((a_perf & a_value & a_inj).mean() * 100, 1), "%")

,table,% joueurs couverts
4,transfer_history,98.4
0,player_performances,95.4
6,player_teammates_played_with,91.1
1,player_market_value,74.9
2,player_latest_market_value,74.9
3,player_injuries,37.3
5,player_national_performances,30.4


Performances + valeur marchande : 74.6 %
Performances + valeur + au moins une blessure : 36.4 %


In [50]:
noms = (tables["player_performances"][["team_id", "team_name"]]
        .drop_duplicates("team_id").set_index("team_id")["team_name"])

top_nat = tables["player_national_performances"]["team_id"].value_counts().head(10)
print(top_nat.to_frame("nb_lignes").join(noms))

         nb_lignes team_name
team_id                     
21746          839       NaN
21100          760       NaN
21426          754       NaN
23133          728       NaN
16371          681       NaN
23243          667       NaN
9067           649       NaN
5710           648       NaN
16373          643       NaN
31269          606       NaN


Aucune equipe de player_national_performances n'est référencée dans TEAM_DETAILS car il s'agit d'une équipe nationale et non d'une équipe de club.

### Intégrité des données :

- `player_profiles` : 100 % des player_id sont uniques et non nuls. Les variables utiles à la modélisation sont quasi complète : id, name, main_position, date_of_birth, nationality, current_club_id, current_club_name. D'autres variables plus contextuelles sont très imcompletes mais non necessaires à la modélisation.

- `team_details` : 100 % des team_id sont uniques et non nuls. L'identité des clubs (nom, pays) est toujours renseignée. Les informations de compétition (competition_id, nom, club_division) manquent conjointement pour 23 % des clubs. Comme la table ne donne qu’une seule ligne par club, le championnat par saison devra de toute façon être repris de `team_competitions_seasons`.

- `team_children` : La table est complète sans aucune valeur manquante. Cependant son usage reste secondaire : rattacher les équipes réserves à leur club parent.

- `team_competitions_seasons` : Les clés club_id et season_id sont complètes. Les informations de compétition manquent conjointement pour 16 % des couples club-saison. C’est la table de référence pour attribuer un championnat et une division à chaque club par saison.


### `player_profiles`

In [51]:
print("Proportion de valeurs manquantes par variables dans la table Player_profiles")
pd.DataFrame({
    "nb" : tables["player_profiles"].isna().sum(),
    "pct" : (tables["player_profiles"].isna().mean() * 100 ).round(2)
}).sort_values("pct")


Proportion de valeurs manquantes par variables dans la table Player_profiles


,nb,pct
player_id,0,0.00
player_slug,0,0.00
player_image_url,0,0.00
current_club_name,0,0.00
current_club_id,0,0.00
main_position,2,0.00
height,4,0.00
is_eu,0,0.00
position,2,0.00
citizenship,57,0.06


In [52]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_profiles"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_profiles"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_profiles"].columns[tables["player_profiles"].isna().all()])


Nombre de lignes avec au moins une valeurs manquante :  92671
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


#### Doublons

In [53]:
cols_hors_id = [c for c in tables["player_profiles"].columns if c not in ("player_id", "player_name", "player_image_url")]

print("Doublons stricts            :", tables["player_profiles"].duplicated().sum())
print("player_id répétés           :", tables["player_profiles"]["player_id"].duplicated().sum())
print("Doublons hors identifiant   :", tables["player_profiles"].duplicated(cols_hors_id).sum())
print("Lignes au slug répété       :", tables["player_profiles"]["player_slug"].duplicated(keep=False).sum())

cle_identite = ["player_slug", "date_of_birth", "citizenship"]
suspects = tables["player_profiles"][tables["player_profiles"].duplicated(cle_identite, keep=False) & tables["player_profiles"]["date_of_birth"].notna()]
print("Lignes même slug + naissance + nationalité :", len(suspects))
display(suspects.sort_values(cle_identite)[
    ["player_id", "player_slug", "date_of_birth", "citizenship", "height",
     "position", "current_club_name", "joined"]])

Doublons stricts            : 0
player_id répétés           : 0
Doublons hors identifiant   : 0
Lignes au slug répété       : 7372
Lignes même slug + naissance + nationalité : 4


,player_id,player_slug,date_of_birth,citizenship,height,position,current_club_name,joined
7825,1096879,felipe-caua,2004-05-06,Brazil,0.0,Defender - Centre-Back,Sampaio Corrêa FC (MA) U20,NaN
9923,1370267,felipe-caua,2004-05-06,Brazil,0.0,Defender - Centre-Back,Sampaio Corrêa FC (MA),NaN
24230,179078,pacheta,1981-09-09,Spain,185.0,Attack - Centre-Forward,Retired,2021-07-01
54946,505313,pacheta,1981-09-09,Spain,0.0,Attack,Retired,NaN


Il semble n'y avoir que deux doublons strictes, car même nom, date de naissance et nationalité. On verra au moment de la joiture la décision à prendre car il ne faudrait pas supprimer un player_id avec des lignes dans les autres tables.

In [54]:
tables["player_profiles"]["suspected_duplicate"] = tables["player_profiles"]["player_id"].isin(suspects["player_id"])

#### Selection des colonnes utiles au sujet 

In [55]:
suppressions = {
    "Hors sujet (aucun lien avec le risque de blessure, de décès ou la valeur)": [
        "player_image_url", "outfitter", "social_media_url",
        "player_agent_id", "player_agent_name"],
    "Redondant ou trop détaillé (name_in_home_country doublonne le nom ; country_of_birth suffit)": [
        "name_in_home_country", "place_of_birth"],
    "Situation actuelle, pas historique : le club par saison vient de player_performances": [
        "joined", "contract_expires", "contract_option", "date_of_last_contract_extension",
        "on_loan_from_club_id", "on_loan_from_club_name", "contract_there_expires"],
    "Plus de 98 % de valeurs manquantes": [
        "second_club_url", "second_club_name", "third_club_url", "third_club_name",
        "fourth_club_url", "fourth_club_name"],
}

for i, (raison, cols) in enumerate(suppressions.items(), start=3):
    try:
        taux = (tables["player_profiles"][cols].isna().mean() * 100).round(1)
        print(f"{raison}\n  {dict(taux)}\n")
    except:
        pass
try:
    tables["player_profiles"] = tables["player_profiles"].drop(columns=[c for cols in suppressions.values() for c in cols])
except:
    pass
print(tables["player_profiles"].shape)

Hors sujet (aucun lien avec le risque de blessure, de décès ou la valeur)
  {'player_image_url': np.float64(0.0), 'outfitter': np.float64(93.6), 'social_media_url': np.float64(78.1), 'player_agent_id': np.float64(58.9), 'player_agent_name': np.float64(53.7)}

Redondant ou trop détaillé (name_in_home_country doublonne le nom ; country_of_birth suffit)
  {'name_in_home_country': np.float64(46.4), 'place_of_birth': np.float64(17.9)}

Situation actuelle, pas historique : le club par saison vient de player_performances
  {'joined': np.float64(3.6), 'contract_expires': np.float64(58.3), 'contract_option': np.float64(96.1), 'date_of_last_contract_extension': np.float64(91.0), 'on_loan_from_club_id': np.float64(95.9), 'on_loan_from_club_name': np.float64(95.9), 'contract_there_expires': np.float64(97.3)}

Plus de 98 % de valeurs manquantes
  {'second_club_url': np.float64(98.9), 'second_club_name': np.float64(98.9), 'third_club_url': np.float64(100.0), 'third_club_name': np.float64(100.0), 'fo

#### Nom du joueur

`player_name` contient l'identifiant entre parenthèses (« Miroslav Klose (10) »). On retire ce suffixe. Les noms manquants sont reconstruits à partir du slug.

In [56]:
motif_suffixe = r"\s*\(\d+\)$"
avec_suffixe = tables["player_profiles"]["player_name"].str.contains(motif_suffixe, na=False)
tables["player_profiles"]["player_name"] = tables["player_profiles"]["player_name"].str.replace(motif_suffixe, "", regex=True).str.strip()

manquant = tables["player_profiles"]["player_name"].isna()
tables["player_profiles"].loc[manquant, "player_name"] = tables["player_profiles"].loc[manquant, "player_slug"].str.replace("-", " ").str.title()

print("Suffixes retirés :", avec_suffixe.sum(), "| noms reconstruits :", manquant.sum())

Suffixes retirés : 92151 | noms reconstruits : 520


#### Date de naissance 


le format actuel est `YYYY-MM-DD`. On peut le convertir en datetime pour calculer l'âge du joueur à chaque saison. Il n'y a pas de date de naissance manquante.

In [57]:
tables["player_profiles"]["date_of_birth"] = pd.to_datetime(tables["player_profiles"]["date_of_birth"], format="%Y-%m-%d", errors="coerce")
erreurs = tables["player_profiles"]["date_of_birth"].isna().sum() - tables["player_profiles"]["date_of_birth"].isna().sum()
print("Valeurs non convertibles :", erreurs)
print("Manquantes :", tables["player_profiles"]["date_of_birth"].isna().sum())
print("Plage :", tables["player_profiles"]["date_of_birth"].min().date(), "→", tables["player_profiles"]["date_of_birth"].max().date())


Valeurs non convertibles : 0
Manquantes : 1006
Plage : 1952-02-13 → 2010-10-12


#### Date de décès

De même on convertit le format en datetime.

In [58]:
motif_deces = r"^(?P<date>\d{2}\.\d{2}\.\d{4}) \((?P<age>\d+)\)$"
extrait = tables["player_profiles"]["date_of_death"].str.extract(motif_deces)
assert extrait["date"].notna().sum() == tables["player_profiles"]["date_of_death"].notna().sum(), "format inattendu"

tables["player_profiles"]["date_of_death"] = pd.to_datetime(extrait["date"], format="%d.%m.%Y")
tables["player_profiles"]["is_dead"] = tables["player_profiles"]["date_of_death"].notna()

# Controle de la concordance des dates
d, n = tables["player_profiles"]["date_of_death"], tables["player_profiles"]["date_of_birth"]
anniv_pas_passe = (d.dt.month < n.dt.month) | ((d.dt.month == n.dt.month) & (d.dt.day < n.dt.day))
age_calcule = d.dt.year - n.dt.year - anniv_pas_passe.astype(int)
concordance = (age_calcule[tables["player_profiles"]["is_dead"]] == extrait["age"][tables["player_profiles"]["is_dead"]].astype(int)).mean()

print("Décès :", tables["player_profiles"]["is_dead"].sum())
print("Plage :", tables["player_profiles"]["date_of_death"].min().date(), "→", tables["player_profiles"]["date_of_death"].max().date())
print(f"Concordance âge indiqué / recalculé : {concordance:.0%}")

Décès : 272
Plage : 2001-08-19 → 2025-08-14
Concordance âge indiqué / recalculé : 100%


#### Statut actuel du joueur





In [59]:
print(tables["player_profiles"]["current_club_id"].value_counts().head(10))
print(tables["player_profiles"]["current_club_name"].value_counts().head(10))

current_club_id
123       23380
515        8599
75         1930
4023        271
2113        102
26370        47
20406        45
21459        44
112192       44
86387        41
Name: count, dtype: int64
current_club_name
Retired                   23380
Without Club               8599
Unknown                    1930
---                         271
Career break                102
Arsenal FC                   58
Botafogo FC                  53
Pro Vercelli Primavera       47
Reggiana Primavera           45
Gangwon FC                   44
Name: count, dtype: int64


In [60]:
pseudo_clubs = (tables["player_profiles"].groupby("current_club_id")
                .agg(libelle=("current_club_name", "unique"),
                     nb_joueurs=("player_id", "size"),
                     nb_decedes=("date_of_death", lambda s: s.notna().sum()))
                .sort_values("nb_joueurs", ascending=False)
                .head(8))
pseudo_clubs

,libelle,nb_joueurs,nb_decedes
current_club_id,,,
123,[Retired],23380,3
515,[Without Club],8599,0
75,[Unknown],1930,1
4023,[---],271,268
2113,[Career break],102,0
26370,[Pro Vercelli Primavera],47,0
20406,[Reggiana Primavera],45,0
21459,[Gangwon FC],44,0


`current_club_id` ne sert pas comme club historique, mais certains identifiants sont des codes de statut : 123 = *Retired*, 515 = *Without Club*, 2113 = *Career break*, 75 = *Unknown*, et 4023 = `---`, qui correspond aux joueurs décédés. On en tire une variable `current_status`, utile pour savoir si un joueur est encore en activité (fin d'exposition au risque), puis on retire les deux colonnes de club.

In [61]:
print(pd.crosstab(tables["player_profiles"]["current_club_id"].eq(4023), tables["player_profiles"]["is_dead"],
                  rownames=["club « --- »"], colnames=["is_dead"]))

codes_statut = {123: "retired", 515: "without_club", 2113: "career_break", 75: "unknown", 4023: "unknown"}
tables["player_profiles"]["current_status"] = tables["player_profiles"]["current_club_id"].map(codes_statut).fillna("active")
tables["player_profiles"].loc[tables["player_profiles"]["is_dead"], "current_status"] = "deceased"
print(tables["player_profiles"]["current_status"].value_counts())

tables["player_profiles"] = tables["player_profiles"].drop(columns=["current_club_id", "current_club_name"])


is_dead       False  True 
club « --- »              
False         92396      4
True              3    268
current_status
active          58389
retired         23377
without_club     8599
unknown          1932
deceased          272
career_break      102
Name: count, dtype: int64


#### Taille

toutes les tailles en dessous de 1,40 m sont suspectes pour de joueurs de foot professionnels. On les considères comme des valeurs manquantes.

In [62]:
print("height = 0 :", (tables["player_profiles"]["height"] == 0).sum(), "| 0 < height < 140 :", tables["player_profiles"]["height"].between(0.1, 139).sum())
aberrant = tables["player_profiles"]["height"] < 140
tables["player_profiles"].loc[aberrant, "height"] = np.nan
print("Plage après :", tables["player_profiles"]["height"].min(), "→", tables["player_profiles"]["height"].max(),
      f"| manquants : {tables["player_profiles"]['height'].isna().mean():.1%}")

height = 0 : 18867 | 0 < height < 140 : 1
Plage après : 152.0 → 210.0 | manquants : 20.4%


#### Nationalité

In [63]:
tables["player_profiles"]["citizenship"] = tables["player_profiles"]["citizenship"].str.replace("Cookinseln", "Cook Islands")
nat = tables["player_profiles"]["citizenship"].str.split(r"\s{2,}", expand=True)
print("Nombre maximal de nationalités :", nat.shape[1])

tables["player_profiles"]["nationality_1"] = nat[0].str.strip()
tables["player_profiles"]["nationality_2"] = nat[1].str.strip()
tables["player_profiles"]["has_dual_nationality"] = tables["player_profiles"]["nationality_2"].notna()
print("Doubles nationalités :", tables["player_profiles"]["has_dual_nationality"].sum())
tables["player_profiles"] = tables["player_profiles"].drop(columns="citizenship")

Nombre maximal de nationalités : 2
Doubles nationalités : 19956


#### Poste

`position` combine le poste général et le poste détaillé (« Defender - Right-Back »). La partie avant le tiret est identique à `main_position` : on ne garde donc que la partie détaillée dans `sub_position`. Pour les joueurs dont seul le poste général est connu (« Mitables["player_profiles"]ield »), `sub_position` reste vide. Le poste détaillé peut compter pour le risque de blessure (un latéral ne court pas comme un défenseur central).

In [64]:
poste = tables["player_profiles"]["position"].str.split(" - ", n=1, expand=True)
assert (poste[0].eq(tables["player_profiles"]["main_position"]) | tables["player_profiles"]["position"].isna()).all()

tables["player_profiles"]["sub_position"] = poste[1]
tables["player_profiles"].loc[tables["player_profiles"]["main_position"] == "Goalkeeper", "sub_position"] = "Goalkeeper"
generique = tables["player_profiles"]["sub_position"].isna() & tables["player_profiles"]["main_position"].notna()
print("Postes sans détail :", generique.sum())
display(pd.crosstab(tables["player_profiles"]["sub_position"].fillna("(non précisé)"), tables["player_profiles"]["main_position"]))

tables["player_profiles"] = tables["player_profiles"].drop(columns="position")

Postes sans détail : 7046


main_position,Attack,Defender,Goalkeeper,Midfield
sub_position,,,,
(non précisé),1885,2013,0,3148
Attacking Midfield,0,0,0,5821
Central Midfield,0,0,0,9546
Centre-Back,0,15074,0,0
Centre-Forward,12070,0,0,0
Defensive Midfield,0,0,0,6999
Goalkeeper,0,0,10276,0
Left Midfield,0,0,0,1314
Left Winger,4756,0,0,0


#### Pied fort


In [65]:
tables["player_profiles"]["foot"] = tables["player_profiles"]["foot"].str.strip().str.lower()
print(tables["player_profiles"]["foot"].value_counts(dropna=False))

foot
right    49186
NaN      23488
left     16870
both      3127
Name: count, dtype: int64


#### Ajencement des variables et controles finaux 

In [66]:
colonnes_finales = [
    "player_id", "player_name", "player_slug",
    "date_of_birth", "country_of_birth", "nationality_1", "nationality_2",
    "has_dual_nationality", "is_eu",
    "height", "foot", "main_position", "sub_position",
    "date_of_death", "is_dead", "current_status", "suspected_duplicate",
]
assert set(colonnes_finales) == set(tables["player_profiles"].columns), set(tables["player_profiles"].columns) ^ set(colonnes_finales)
propre_profiles = tables["player_profiles"][colonnes_finales].copy()

for c in ["foot", "main_position", "sub_position", "current_status"]:
    propre_profiles[c] = propre_profiles[c].astype("category")

propre_profiles.dtypes

player_id                        int64
player_name                        str
player_slug                        str
date_of_birth           datetime64[us]
country_of_birth                   str
nationality_1                      str
nationality_2                      str
has_dual_nationality              bool
is_eu                             bool
height                         float64
foot                          category
main_position                 category
sub_position                  category
date_of_death           datetime64[us]
is_dead                           bool
current_status                category
suspected_duplicate               bool
dtype: object

In [67]:
assert propre_profiles["player_id"].is_unique and propre_profiles["player_id"].notna().all()
assert propre_profiles["date_of_birth"].dropna().between("1950-01-01", "2012-12-31").all()
assert propre_profiles["height"].dropna().between(140, 215).all()
assert (propre_profiles["is_dead"] == propre_profiles["date_of_death"].notna()).all()
assert (propre_profiles["date_of_death"].dropna() > propre_profiles.loc[propre_profiles["is_dead"], "date_of_birth"]).all()
assert (propre_profiles["has_dual_nationality"] == propre_profiles["nationality_2"].notna()).all()
assert not propre_profiles["player_name"].str.contains(r"\(\d+\)$").any()
print("Tous les contrôles passent.")

resume = pd.DataFrame({
    "type": propre_profiles.dtypes.astype(str),
    "% manquants": (propre_profiles.isna().mean() * 100).round(2),
    "nb distincts": propre_profiles.nunique(),
})
print(f"{tables["player_profiles"].shape} → {propre_profiles.shape}")

Tous les contrôles passent.
(92671, 17) → (92671, 17)


### `team_details`

In [68]:
print(tables["team_details"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_details")
pd.DataFrame({
    "nb" : tables["team_details"].isna().sum(),
    "pct" : (tables["team_details"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(2175, 12)
Proportion de valeurs manquantes par variables dans la table team_details


,nb,pct
club_id,0,0.00
club_slug,0,0.00
club_name,0,0.00
logo_url,0,0.00
country_name,0,0.00
season_id,0,0.00
source_url,0,0.00
_last_modified_at,0,0.00
competition_id,506,23.26
competition_slug,506,23.26


In [69]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_details"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_details"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_details"].columns[tables["team_details"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  506
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


#### Doublons

In [70]:
print("Doublons stricts :", tables["team_details"].duplicated().sum())
print("club_id répétés :", tables["team_details"]["club_id"].duplicated().sum())
print("club_slug répétés :", tables["team_details"]["club_slug"].duplicated().sum())
print("club_name répétés :", tables["team_details"]["club_name"].duplicated().sum())


Doublons stricts : 0
club_id répétés : 0
club_slug répétés : 0
club_name répétés : 0


#### Saison et compétition

Chaque ligne de `team_details` vient de la saison de référence du club de la page Transfermarkt correspondante (le `saison_id` est celui `source_url` qui est la page d'où viennent les informations). Elle change d'un club à l'autre, entre 2000 et 2025. La compétition (`competition_id`, `competition_name`, `club_division`) n'est donc valable que pour cette saison-là, et ne décrit pas le club en général.

On vérifie si les informations de compétition sont cohérentes avec la table `team_competitions_seasons` afin d'être sur de ne pouvoir utiliser que la table `team_competitions_seasons`

In [71]:
saison_url = tables["team_details"]["source_url"].str.extract(r"saison_id/(\d+)")[0].astype(int)
print("season_id = saison de source_url :", f"{(saison_url == tables["team_details"]['season_id']).mean():.0%}")
print("Plage de season_id :", tables["team_details"]["season_id"].min(), "→", tables["team_details"]["season_id"].max(),
      "|", tables["team_details"]["season_id"].nunique(), "valeurs différentes")

renseigne = tables["team_details"]["competition_name"].notna()
print("club_division = competition_name :",
      f"{(tables["team_details"].loc[renseigne, 'club_division'] == tables["team_details"].loc[renseigne, 'competition_name']).mean():.0%}")

# La compétition de team_details se retrouve-t-elle dans tcs pour le même couple (club, saison) ?
comp_tcs = (tables["team_competitions_seasons"].dropna(subset=["competition_id"])
               .groupby(["club_id", "season_id"])["competition_id"].apply(set))
def retrouvee(club, saison, comp):
    cle = (club, saison)
    if pd.isna(comp):
            return cle not in comp_tcs.index
    return cle in comp_tcs.index and comp in comp_tcs[cle]

concordance = pd.Series([retrouvee(*t) for t in zip(tables["team_details"]["club_id"], tables["team_details"]["season_id"], tables["team_details"]["competition_id"])])
print(f"Compétition cohérente avec team_competitions_seasons : {concordance.mean():.2%}")
display(tables["team_details"].loc[~concordance.values, ["club_id", "club_name", "season_id", "competition_id", "competition_name"]])

season_id = saison de source_url : 100%
Plage de season_id : 2000 → 2025 | 26 valeurs différentes
club_division = competition_name : 100%
Compétition cohérente avec team_competitions_seasons : 99.95%


,club_id,club_name,season_id,competition_id,competition_name
1140,34232,Muaither SC (34232),2020,QU2L,Qatari Second Division


Les colonnes de compétition sont redondantes avec `team_competitions_seasons` pour 99,95 % des clubs, et ne couvrent qu'une saison arbitraire. On les retire, avec `season_id`. Le seul écart (Muaither SC, 2020) est un couple club-saison sans compétition dans `team_competitions_seasons`.

On retire aussi les colonnes techniques (`logo_url`, `source_url`, `_last_modified_at`).

In [72]:
cols_competition = ["season_id", "competition_id", "competition_slug", "competition_name", "club_division"]
cols_techniques = ["logo_url", "source_url", "_last_modified_at"]

print(tables["team_details"].shape)

(2175, 12)


#### Nom du club

Comme dans `player_profiles`, `club_name` se termine par l'identifiant entre parenthèses (« CD Mafra U23 (100335) »). On retire ce suffixe.

In [73]:
motif_suffixe = r"\s*\(\d+\)$"
avec_suffixe = tables["team_details"]["club_name"].str.contains(motif_suffixe)
tables["team_details"]["club_name"] = tables["team_details"]["club_name"].str.replace(motif_suffixe, "", regex=True).str.strip()

tronques = tables["team_details"]["club_name"].str.endswith("\\")
display(tables["team_details"].loc[tronques, ["club_id", "club_name", "club_slug"]])

,club_id,club_name,club_slug
189,11374,Jeanne d\,jeanne-darc-de-drancy
324,1286,CA Newell\,club-atletico-newells-old-boys
548,17707,US Jeanne d\,us-jeanne-darc-carquefou
551,17665,Les Genêts d\,genets-anglet
740,23206,ES Pays d\,es-pays-duzes
864,2618,Olympique d\,olympique-ales
921,27490,C\,cchartres-football
1354,4153,L\,laquila-calcio
1933,7882,US Raon-l\,us-raon-letape


Des noms se sont cassés lors de l'extraction, sûrement dû à une apostrophe, on les reconstruit à la main à partir du slug.

In [74]:
corrections = {
    11374: "Jeanne d'Arc de Drancy",
    1286:  "CA Newell's Old Boys",
    17707: "US Jeanne d'Arc Carquefou",
    17665: "Les Genêts d'Anglet",
    23206: "ES Pays d'Uzès",
    2618:  "Olympique d'Alès",
    27490: "C'Chartres Football",
    4153:  "L'Aquila Calcio",
    7882:  "US Raon-l'Étape",
}
assert set(corrections) == set(tables["team_details"].loc[tronques, "club_id"]), "liste des noms tronqués différente"
tables["team_details"]["club_name"] = tables["team_details"]["club_id"].map(corrections).fillna(tables["team_details"]["club_name"])

print("Suffixes retirés :", avec_suffixe.sum(), "| noms corrigés :", tronques.sum())
homonymes = tables["team_details"][tables["team_details"]["club_name"].duplicated(keep=False)]
print("Homonymes après retrait du suffixe :")
display(homonymes[["club_id", "club_name", "country_name"]])

Suffixes retirés : 2166 | noms corrigés : 9
Homonymes après retrait du suffixe :


,club_id,club_name,country_name
142,11,Arsenal FC,England
1447,4673,Arsenal FC,Argentina


#### Equipes de jeunes et réserves

La table mélange équipes premières et équipes de jeunes ou réserves (U17, U21, Primavera, équipes B…). Le produit vise les **clubs professionnels**, et un match en U19 n'expose pas au même risque qu'un match en équipe première. On crée donc un indicateur `is_youth_or_reserve`, à partir du nom :

- mention d'âge ou de catégorie n'importe où dans le nom : `U21`, `Under 17`, `Sub-23`, `Primavera`, `Youth`, `Academy`, `Reserves`, ou `Jong` en début de nom ;
- suffixe d'équipe réserve en fin de nom : ` II`, ` III`, ` B`, ` C`, ` 2`.

Le suffixe n'est testé qu'en fin de nom pour ne pas toucher des clubs comme *Willem II Tilburg*. On ne supprime aucune ligne : c'est un indicateur.

In [75]:
motif_categorie = r"(?:\bU\s?-?\d{2}\b|\bUnder\s?\d{2}\b|\bSub-?\d{2}\b|\bPrimavera\b|\bJuvenil\b|\bYouth\b|\bAcademy\b|\bReserves?\b|^Jong\s)"
motif_reserve = r"\s(?:II|III|B|C|2)$"
nom_sans_date = tables["team_details"]["club_name"].str.replace(r"\s*\(-\s?\d{4}\)$", "", regex=True)   # « (-2022) » = club disparu

tables["team_details"]["is_youth_or_reserve"] = (nom_sans_date.str.contains(motif_categorie)
                             | nom_sans_date.str.contains(motif_reserve))
print("Équipes de jeunes ou réserves :", tables["team_details"]["is_youth_or_reserve"].sum(), "sur", len(tables["team_details"]))
print(tables["team_details"].loc[tables["team_details"]["is_youth_or_reserve"], "club_name"].sample(15, random_state=0).tolist())


ligue_jeunes = tables["team_details"]["club_division"].str.contains(r"Under|U\d{2}|Primavera|Premier League 2|Next Pro|Revela", na=False)
display(tables["team_details"].loc[ligue_jeunes & ~tables["team_details"]["is_youth_or_reserve"], ["club_id", "club_name"]]
          .assign(ligue=tables["team_details"]["club_division"]))

Équipes de jeunes ou réserves : 302 sur 2175
['Torino Under 18', 'Chievo Under 17', 'Genoa Under 18', 'AC Sparta Prague B', 'Livorno Primavera', 'Reggiana Under 17', 'Como Under 17', 'Cesena Under 17', 'FC Famalicão U23', 'Crystal Palace U21', 'Ipswich Town U21', 'Jong KAA Gent', 'Juve Stabia Under 17', 'Austin FC II', 'AJ Auxerre B']


,club_id,club_name,ligue
5,101282,Crown Legacy FC,MLS Next Pro
46,103649,Huntsville City FC,MLS Next Pro
53,103657,Carolina Core FC,MLS Next Pro
1103,33293,Chattanooga FC,MLS Next Pro
1127,33285,Ventura County FC,MLS Next Pro
1424,45806,Real Monarchs SLC,MLS Next Pro
1446,46720,Tacoma Defiance,MLS Next Pro
1834,71021,North Texas SC,MLS Next Pro
2114,95338,The Town FC,MLS Next Pro


Je ne suis pas sûr que les clubs restants soient des réserves ou des équipes indépendantes. Pour l'instant on ne touche pas, on pourra vérifier plus tard à partir de `team_children` qui permettrait de completer l'indicateur.

#### Pays

`country_name` est toujours renseigné. On vérifie juste que le format est le même que dans `player_profiles`, pour pouvoir comparer nationalité du joueur et pays du club.



In [76]:
print(tables["team_details"]["country_name"].nunique(), "pays")
print(tables["team_details"]["country_name"].value_counts().head(10))

38 pays
country_name
Italy            345
France           212
England          122
Spain            119
United States    108
Türkiye          103
Portugal          92
Germany           92
Mexico            87
Belgium           84
Name: count, dtype: int64


#### Controles finaux

In [77]:
assert tables["team_details"]["club_id"].is_unique and tables["team_details"]["club_id"].notna().all()
assert tables["team_details"]["club_name"].notna().all()
assert not tables["team_details"]["club_name"].str.contains(r"\(\d+\)$|\\$").any()
assert tables["team_details"]["country_name"].notna().all()
print("Tous les contrôles passent.")
print(f"{tables["team_details"].shape} → {tables["team_details"].shape}")

Tous les contrôles passent.
(2175, 13) → (2175, 13)


### `team_children`

In [78]:
print(tables["team_children"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_children")
pd.DataFrame({
    "nb" : tables["team_children"].isna().sum(),
    "pct" : (tables["team_children"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(7695, 5)
Proportion de valeurs manquantes par variables dans la table team_children


,nb,pct
parent_team_id,0,0.0
parent_team_name,0,0.0
child_team_id,0,0.0
child_team_name,0,0.0
_last_modified_at,0,0.0


In [79]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_children"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_children"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_children"].columns[tables["team_children"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  0
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_competitions_seasons`

In [80]:
print(tables["team_competitions_seasons"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_competitions_seasons")
pd.DataFrame({
    "nb" : tables["team_competitions_seasons"].isna().sum(),
    "pct" : (tables["team_competitions_seasons"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(196378, 7)
Proportion de valeurs manquantes par variables dans la table team_competitions_seasons


,nb,pct
club_id,0,0.00
team_name,0,0.00
season_id,0,0.00
_last_modified_at,0,0.00
competition_name,32189,16.39
competition_id,32189,16.39
club_division,32189,16.39


In [81]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_competitions_seasons"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_competitions_seasons"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_competitions_seasons"].columns[tables["team_competitions_seasons"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  32189
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


#### Doublons

In [82]:
nb_doublons = tables["team_competitions_seasons"].duplicated().sum()
print(f"Doublons stricts : {nb_doublons} sur {len(tables["team_competitions_seasons"])} lignes ({nb_doublons / len(tables["team_competitions_seasons"]):.0%})")

copies = tables["team_competitions_seasons"].groupby(list(tables["team_competitions_seasons"].columns), dropna=False).size()
print("Nombre de copies par ligne distincte :")
print(copies.value_counts().sort_index().to_string())

tcs = tables["team_competitions_seasons"].drop_duplicates().reset_index(drop=True)
print("Après suppression :", tcs.shape)
print("Couple (club_id, season_id) unique :", not tcs.duplicated(["club_id", "season_id"]).any())

Doublons stricts : 143359 sur 196378 lignes (73%)
Nombre de copies par ligne distincte :
1      5551
2      8038
3     10778
4     13042
5      8428
6      3885
7      2072
8       869
9       258
10       69
11       22
12        3
13        3
14        1
Après suppression : (53019, 7)
Couple (club_id, season_id) unique : True


Environ trois quarts des lignes sont des copies exactes : chaque couple club-saison apparaît entre 1 à 14 fois, avec la `_last_modified_at`. Ce ne sont des doublons de scraping. Une fois retirés, il reste exactement une ligne par club et par saison.

#### Compétition non changeante 

In [83]:
nb_comp = tcs.groupby("club_id")["competition_id"].nunique(dropna=False)
print("Nombre de compétitions différentes par club :", nb_comp.value_counts().to_dict())

saisons_gb1 = tcs[tcs["competition_id"] == "GB1"].groupby("season_id")["club_id"].nunique()
print("Clubs en GB1 par saison :", saisons_gb1.unique())
print("Mêmes 20 clubs chaque saison :",
      tcs[tcs["competition_id"] == "GB1"].groupby("season_id")["club_id"].apply(frozenset).nunique() == 1)

Nombre de compétitions différentes par club : {1: 2175}
Clubs en GB1 par saison : [20]
Mêmes 20 clubs chaque saison : True


Chaque club a **une seule compétition** sur toutes ses saisons, et la Premier League compte les 20 mêmes clubs de 2000 à 2025. La table donne donc le **championnat actuel** du club (au moment du scraping), recopié sur chaque saison.

On le confirme avec deux clubs promus et relégués, en comparant avec `player_performances`, qui donne les compétitions réellement jouées par saison. Les saisons y sont notées « 18/19 » : `season_id` 2018 correspond à la saison 2018-19.

In [84]:
perf = tables["player_performances"]
exemples = {41: "Hamburger SV", 897: "Deportivo de La Coruña"}
perf_ex = perf.loc[perf["team_id"].isin(list(exemples)), ["team_id", "season_name", "competition_name"]]

def saison_courte(annee):
    return f"{annee % 100:02d}/{(annee + 1) % 100:02d}"

lignes = []
for club_id, nom in exemples.items():
    actuelle = tcs.loc[tcs["club_id"] == club_id, "competition_name"].iloc[0]
    for annee in range(2017, 2026):
        jouees = perf_ex.loc[(perf_ex["team_id"] == club_id) & (perf_ex["season_name"] == saison_courte(annee)),
                             "competition_name"].drop_duplicates().sort_values()
        lignes.append({"club": nom, "season_id": annee,
                       "team_competitions_seasons": actuelle,
                       "player_performances (compétitions jouées)": ", ".join(jouees)})
pd.DataFrame(lignes)

,club,season_id,team_competitions_seasons,player_performances (compétitions jouées)
0,Hamburger SV,2017,Bundesliga,"Bundesliga, DFB-Pokal"
1,Hamburger SV,2018,Bundesliga,"2. Bundesliga, DFB-Pokal"
2,Hamburger SV,2019,Bundesliga,"2. Bundesliga, DFB-Pokal"
3,Hamburger SV,2020,Bundesliga,"2. Bundesliga, DFB-Pokal"
4,Hamburger SV,2021,Bundesliga,"2. Bundesliga, Bundesliga Playoffs, DFB-Pokal"
5,Hamburger SV,2022,Bundesliga,"2. Bundesliga, Bundesliga Playoffs, DFB-Pokal"
6,Hamburger SV,2023,Bundesliga,"2. Bundesliga, DFB-Pokal"
7,Hamburger SV,2024,Bundesliga,"2. Bundesliga, DFB-Pokal"
8,Hamburger SV,2025,Bundesliga,"Bundesliga, DFB-Pokal"
9,Deportivo de La Coruña,2017,LaLiga2,"Copa del Rey, LaLiga"


#### Données redondantes

- `club_division` : on vérifie qu'elle est identique à `competition_name`.
- `team_name` : on vérifie qu'elle est identique au nom de `team_details` (avec le suffixe « (id) » et les 9 noms tronqués). Le nom propre est dans `team_details` nettoyée.
- `_last_modified_at` : horodatage technique.

In [85]:
td = tables["team_details"]
renseigne = tcs["competition_name"].notna()
print("club_division = competition_name :",
      f"{(tcs.loc[renseigne, 'club_division'] == tcs.loc[renseigne, 'competition_name']).mean():.0%}")


#PROBLEME A CHANGER


noms = tcs.drop_duplicates("club_id").set_index("club_id")["team_name"]
print("team_name = club_name de team_details :",
      f"{(noms == td.set_index('club_id')['club_name'].reindex(noms.index)).mean():.0%}")

club_division = competition_name : 100%
team_name = club_name de team_details : 0%


#### Une ligne par club

Comme la compétition est constante pour chaque club, on garde une ligne par club.

On regarde aussi si les `competition_id` sont les mêmes que dans `team_details`

In [86]:
par_club = (tcs.sort_values(["club_id", "season_id"]).drop_duplicates("club_id")[["club_id", "competition_id", "competition_name"]]
               .rename(columns={"competition_id": "current_competition_id",
                                "competition_name": "current_competition_name"})
               .reset_index(drop=True))
print(par_club.shape)

# Même compétition que dans team_details ?
comp_td = td.set_index("club_id")["competition_id"].reindex(par_club["club_id"]).values
egal = (par_club["current_competition_id"].values == comp_td) | (par_club["current_competition_id"].isna().values & pd.isna(comp_td))
print(f"Compétition identique à celle de team_details : {egal.mean():.0%}")

(2175, 3)
Compétition identique à celle de team_details : 100%


#### Compétitions manquantes

La compétition manque pour 506 clubs (23 %), les mêmes que dans `team_details`. Ce ne sont pas des trous aléatoires : un club est soit toujours renseigné, soit jamais. Ces clubs ne jouent pas, en 2025-2026, dans l'une des 168 compétitions couvertes par le jeu de données.

In [87]:
manquant = par_club["current_competition_id"].isna()
print("Clubs sans compétition :", manquant.sum(), f"({manquant.mean():.0%})")
part_par_club = tcs.groupby("club_id")["competition_id"].apply(lambda s: s.isna().mean())
print("Clubs partiellement renseignés :", ((part_par_club > 0) & (part_par_club < 1)).sum())

Clubs sans compétition : 506 (23%)
Clubs partiellement renseignés : 0


#### Noms de compétition ambigus

Quatre noms de compétition correspondent à deux identifiants différents : par exemple « Bundesliga » en Allemagne et en Autriche. On garde `current_competition_id` comme clé, et on ajoute le pays de la compétition, déduit du pays majoritaire de ses clubs dans `team_details`.

In [88]:
ambigus = par_club.dropna().groupby("current_competition_name")["current_competition_id"].unique()
display(ambigus[ambigus.str.len() > 1])

pays = (par_club.merge(td[["club_id", "country_name"]], on="club_id")
                .dropna(subset=["current_competition_id"])
                .groupby("current_competition_id")["country_name"]
                .agg(lambda s: s.mode().iloc[0]))
par_club["current_competition_country"] = par_club["current_competition_id"].map(pays)

current_competition_name
1.Division             [DK2, RU2]
Bundesliga               [L1, A1]
Premier Liga          [RU1, UKR1]
Regionalliga West    [RLW3, ATRW]
Name: current_competition_id, dtype: object

#### Contrôles finaux 

In [89]:
propre = par_club[["club_id", "current_competition_id", "current_competition_name", "current_competition_country"]].copy()
for c in ["current_competition_id", "current_competition_name", "current_competition_country"]:
    propre[c] = propre[c].astype("category")

assert propre["club_id"].is_unique
assert set(propre["club_id"]) == set(td["club_id"]), "clubs différents de team_details"
assert (propre["current_competition_id"].isna() == propre["current_competition_name"].isna()).all()
assert (propre["current_competition_id"].isna() == propre["current_competition_country"].isna()).all()
assert propre.groupby("current_competition_id", observed=True)["current_competition_name"].nunique().eq(1).all()
print("Tous les contrôles passent.")
print(f"{tables["team_competitions_seasons"].shape} → {propre.shape}")

Tous les contrôles passent.
(196378, 7) → (2175, 4)


### `PLAYER_TEAMMATES_PLAYED_WITH`

In [90]:
print(tables["player_teammates_played_with"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_teammates_played_with")
pd.DataFrame({
    "nb" : tables["player_teammates_played_with"].isna().sum(),
    "pct" : (tables["player_teammates_played_with"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1257342, 6)
Proportion de valeurs manquantes par variables dans la table player_teammates_played_with


,nb,pct
player_id,0,0.00
teammate_player_id,0,0.00
teammate_player_name,0,0.00
ppg_played_with,37080,2.95
minutes_played_with,804153,63.96
joint_goal_participation,1035177,82.33


In [91]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_teammates_played_with"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_teammates_played_with"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_teammates_played_with"].columns[tables["player_teammates_played_with"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1232285
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.


In [96]:
t = tables["player_teammates_played_with"]
print("Doublons stricts :", t.duplicated().sum())
print(
    "Paires joueur-coéquipier répétées :",
    t.duplicated(subset=["player_id", "teammate_player_id"]).sum()
)

Doublons stricts : 0
Paires joueur-coéquipier répétées : 0


Dans player_teammates_played_with : le couple (player_id, teammate_id) est-il unique, ou un même coéquipier apparaît-il plusieurs fois (par exemple s'ils ont joué ensemble dans plusieurs clubs différents) ?

In [93]:
nb_doublons_paires = (
    tables["player_teammates_played_with"]
    .duplicated(subset=["player_id", "teammate_player_id"])
    .sum()
)
print("Nombre de paires (joueur, coéquipier) répétées :", nb_doublons_paires)

Nombre de paires (joueur, coéquipier) répétées : 0


In [97]:
profils = tables["player_profiles"]
ids = set(profils["player_id"])

print("player_id sans profil :", (~t["player_id"].isin(ids)).sum())
print("teammate_player_id sans profil :", (~t["teammate_player_id"].isin(ids)).sum())
print("Joueur associé à lui-même :", t["player_id"].eq(t["teammate_player_id"]).sum())

player_id sans profil : 0
teammate_player_id sans profil : 404920
Joueur associé à lui-même : 0


Le point à éclaircir avant de continuer : 404 920 teammate_player_id ne sont pas retrouvés dans player_profiles. 

In [98]:
t = tables["player_teammates_played_with"]
profils = tables["player_profiles"]
ids = set(profils["player_id"])

orphelins = t.loc[~t["teammate_player_id"].isin(ids)]

print("Types d'identifiants :")
print("  teammate_player_id :", t["teammate_player_id"].dtype)
print("  player_id profils :", profils["player_id"].dtype)
print("Lignes concernées :", len(orphelins), f"({len(orphelins) / len(t):.1%})")
print("Identifiants distincts concernés :", orphelins["teammate_player_id"].nunique())

print("Noms les plus fréquents parmi les identifiants absents :")
display(orphelins["teammate_player_name"].value_counts().head(20))

print("Exemples de lignes :")
display(orphelins.sample(min(20, len(orphelins)), random_state=0))

Types d'identifiants :
  teammate_player_id : int64
  player_id profils : int64
Lignes concernées : 404920 (32.2%)
Identifiants distincts concernés : 99075
Noms les plus fréquents parmi les identifiants absents :


teammate_player_name
Pedrinho         105
Paulinho          98
Joãozinho         85
João Paulo        80
Jonathan          75
Vitinho           70
Pedro Bambu       69
Wanderson         68
Ian Mackay        68
Ralf              67
Jean              67
Guilherme         65
Fábio Bahia       63
Cesinha           63
Lucão             63
João Victor       62
Jefferson         61
Juninho           60
Álvaro García     59
Guram Kashia      59
Name: count, dtype: int64

Exemples de lignes :


,player_id,teammate_player_id,teammate_player_name,ppg_played_with,joint_goal_participation,minutes_played_with
1032989,795139,395852,Berkan Turan,1.41,NaN,784.0
845420,621227,621241,Antonio Tricoli,NaN,NaN,34.0
685853,500533,463286,Alberto Fernández,1.52,NaN,445.0
662032,47696,144486,Ertuğrul Şimşek,0.60,NaN,264.0
1035683,796973,577261,Erik De Los Santos,1.79,NaN,NaN
15337,102046,381633,Giorgio Gagliardi,1.22,NaN,NaN
606412,432291,638954,Zoárd Nagy,1.30,1.0,NaN
859833,631821,167722,Vitaliy Buyalskyi,1.89,20.0,NaN
223546,145595,453534,Michel Ferreira,1.37,NaN,NaN
960746,708498,1061125,Yoha Vásquez,1.67,NaN,413.0


In [99]:
colonnes_num = [
    "ppg_played_with",
    "joint_goal_participation",
    "minutes_played_with",
]

display(t[colonnes_num].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
).T)

print("Valeurs négatives :")
display(t[colonnes_num].lt(0).sum())

print("Valeurs égales à zéro :")
display(t[colonnes_num].eq(0).sum())

,count,mean,std,min,50%,90%,95%,99%,max
ppg_played_with,1220262.0,1.480522,0.468990,0.02,1.48,2.03,2.24,3.0,3.0
joint_goal_participation,222165.0,1.949740,1.955698,1.00,1.00,4.00,5.00,10.0,80.0
minutes_played_with,453189.0,451.992837,306.267040,0.00,438.00,888.00,945.00,989.0,999.0


Valeurs négatives :


ppg_played_with             0
joint_goal_participation    0
minutes_played_with         0
dtype: int64

Valeurs égales à zéro :


ppg_played_with                0
joint_goal_participation       0
minutes_played_with         2991
dtype: int64

In [100]:
colonnes_exemple = [
    "player_id",
    "teammate_player_id",
    "teammate_player_name",
    "ppg_played_with",
    "joint_goal_participation",
    "minutes_played_with",
]

print("Minutes exactement égales à 999 :", t["minutes_played_with"].eq(999).sum())
display(t.loc[t["minutes_played_with"].eq(999), colonnes_exemple].head(15))

print("Minutes égales à 0 :", t["minutes_played_with"].eq(0).sum())
display(t.loc[t["minutes_played_with"].eq(0), colonnes_exemple].head(15))

print("Participations aux buts >= 20 :", t["joint_goal_participation"].ge(20).sum())
display(t.nlargest(15, "joint_goal_participation")[colonnes_exemple])

Minutes exactement égales à 999 : 421


,player_id,teammate_player_id,teammate_player_name,ppg_played_with,joint_goal_participation,minutes_played_with
5050,1005843,929574,Freddie Issaka,1.36,2.0,999.0
5623,1006410,1131161,Robinio Vaz,1.35,NaN,999.0
6756,1008031,176265,Berk Ünsal,2.35,NaN,999.0
7873,1009285,783878,Christian Molina,1.52,NaN,999.0
10785,1012709,964561,Ismeal Kabia,1.62,NaN,999.0
13433,1017774,1017775,Mustafa Genç,1.12,NaN,999.0
13523,1017914,1045195,Bradley Slade,1.86,NaN,999.0
19753,1027222,534492,Anthony Civet,1.21,NaN,999.0
20921,1028615,185743,Valentin Sanson,2.35,NaN,999.0
21359,102923,66340,Lazarus Kaimbi,1.11,1.0,999.0


Minutes égales à 0 : 2991


,player_id,teammate_player_id,teammate_player_name,ppg_played_with,joint_goal_participation,minutes_played_with
646,1001055,973567,Gabryel Freitas,3.0,NaN,0.0
652,1001055,916686,Lucão,3.0,NaN,0.0
656,1001055,1066523,Ricardo Cerqueira,3.0,NaN,0.0
657,1001055,1075066,André Dias,3.0,NaN,0.0
658,1001055,1075067,Marco Antônio,3.0,NaN,0.0
659,1001055,1075071,Lucas Evangelista,3.0,NaN,0.0
945,1001411,133480,Alessandro Cerigioni,1.0,NaN,0.0
1272,1001791,885857,Lorenzo Carraro,NaN,NaN,0.0
1273,1001791,885859,Moustafa Yabre,NaN,NaN,0.0
1472,1001940,1106361,Theo Rigopoulos,1.5,NaN,0.0


Participations aux buts >= 20 : 233


,player_id,teammate_player_id,teammate_player_name,ppg_played_with,joint_goal_participation,minutes_played_with
533009,38253,58358,Thomas Müller,2.37,80.0,NaN
795881,58358,38253,Robert Lewandowski,2.37,80.0,NaN
279426,18922,8198,Cristiano Ronaldo,2.27,73.0,NaN
1074608,8198,18922,Karim Benzema,2.27,73.0,NaN
396398,28003,68290,Neymar,2.30,67.0,NaN
922878,68290,28003,Lionel Messi,2.30,67.0,NaN
200556,132098,91845,Heung-min Son,1.78,58.0,NaN
1165180,91845,132098,Harry Kane,1.78,58.0,NaN
396395,28003,65278,Pedro,2.31,54.0,NaN
478683,342229,68290,Neymar,2.25,54.0,NaN


In [101]:
minutes_zero = t["minutes_played_with"].eq(0)

print("Lignes à 0 minute :", minutes_zero.sum())
print("Avec ppg renseigné :", t.loc[minutes_zero, "ppg_played_with"].notna().sum())
print(
    "Avec participation aux buts renseignée :",
    t.loc[minutes_zero, "joint_goal_participation"].notna().sum()
)

display(
    t.loc[minutes_zero, colonnes_exemple]
    .sample(min(20, minutes_zero.sum()), random_state=0)
)

Lignes à 0 minute : 2991
Avec ppg renseigné : 1953
Avec participation aux buts renseignée : 2


,player_id,teammate_player_id,teammate_player_name,ppg_played_with,joint_goal_participation,minutes_played_with
160620,1226628,50144,Cássio,3.0,NaN,0.0
197278,1313626,1308918,Mattia Balzano,1.0,NaN,0.0
170089,1248622,1178304,Andrea Borserio Gaggi,1.5,NaN,0.0
208052,1356182,170982,Jonathan Beaulieu,3.0,NaN,0.0
202216,1329899,610640,Marek Michalicka,NaN,NaN,0.0
206450,1348383,964763,Mehmet Yağmur,3.0,NaN,0.0
1144902,896446,644270,Chapa Herrera,NaN,NaN,0.0
1148827,900153,786987,Yota Komi,3.0,NaN,0.0
204086,1339073,964763,Mehmet Yağmur,NaN,NaN,0.0
117349,1146281,539369,Romarinho,3.0,NaN,0.0


Aucun doublon strict ni doublon de paire joueur-coéquipier détecté. Aucune valeur numérique négative. Les valeurs 0 et 999 de minutes_played_with sont conservées en l’absence de définition confirmant qu’elles codent des valeurs manquantes ou plafonnées. Les références de coéquipiers sans profil correspondant sont documentées; les lignes sont conservées.

### `PLAYER INJURY HISTORIES`

In [327]:
print(tables["player_injuries"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_injuries")
pd.DataFrame({
    "nb" : tables["player_injuries"].isna().sum(),
    "pct" : (tables["player_injuries"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(143195, 7)
Proportion de valeurs manquantes par variables dans la table player_injuries


,nb,pct
player_id,0,0.00
season_name,0,0.00
injury_reason,0,0.00
games_missed,0,0.00
from_date,22,0.02
days_missed,22,0.02
end_date,1523,1.06


In [328]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_injuries"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_injuries"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_injuries"].columns[tables["player_injuries"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1545
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.


HYPER IMPORTANT DE S'ATTARDER LÀ DESSUS

In [329]:
print("Doublons stricts :", tables["player_injuries"].duplicated().sum())

Doublons stricts : 111


In [330]:
lignes_par_joueur = tables["player_injuries"]["player_id"].value_counts()
print("Statistiques du nombre de blessures par joueur :")
print(lignes_par_joueur.describe())

Statistiques du nombre de blessures par joueur :
count    34561.000000
mean         4.143254
std          3.954910
min          1.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         15.000000
Name: count, dtype: float64


days_missed et games_missed ont-ils des valeurs négatives impossibles ?


In [331]:
print(tables["player_injuries"][["days_missed", "games_missed"]].describe())
print()
print(tables["player_injuries"]["injury_reason"].value_counts().head(10))

         days_missed   games_missed
count  143173.000000  143195.000000
mean       51.719291       6.628479
std       104.437179       9.468380
min         1.000000       0.000000
25%        10.000000       1.000000
50%        22.000000       3.000000
75%        53.000000       7.000000
max      8655.000000     208.000000

injury_reason
unknown injury            27028
Muscle injury              6433
Hamstring injury           5786
Knee injury                5544
muscular problems          4880
Ankle injury               4602
Cruciate ligament tear     4035
Ill                        3694
Corona virus               3469
Knock                      2991
Name: count, dtype: int64


### `PLAYER_NATIONAL_TEAM_PERFORMANCES`

In [332]:
print(tables["player_national_performances"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_national_performances")
pd.DataFrame({
    "nb" : tables["player_national_performances"].isna().sum(),
    "pct" : (tables["player_national_performances"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(92701, 9)
Proportion de valeurs manquantes par variables dans la table player_national_performances


,nb,pct
player_id,0,0.00
team_id,0,0.00
matches,0,0.00
goals,0,0.00
shirt_number,0,0.00
career_state,0,0.00
debut_game_id,30347,32.74
coach_id,36922,39.83
debut,92701,100.00


In [333]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_national_performances"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_national_performances"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_national_performances"].columns[tables["player_national_performances"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  92701
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index(['debut'], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.

In [334]:
print("Doublons stricts :", tables["player_national_performances"].duplicated().sum())

Doublons stricts : 32


Les statistiques de matchs, buts, ont-elles des anomalies (ex : but negatif ?)

In [335]:
print(
    tables["player_national_performances"][
        ["matches", "goals"]
    ].describe()
)

            matches         goals
count  92701.000000  92701.000000
mean       7.336027      0.911382
std       12.943084      2.910156
min        0.000000      0.000000
25%        1.000000      0.000000
50%        4.000000      0.000000
75%        8.000000      1.000000
max      223.000000    141.000000


### `PLAYER_TRANSFERT_HISTORIES`

In [336]:
print(tables["transfer_history"].shape)
print("Proportion de valeurs manquantes par variables dans la table transfer_history")
pd.DataFrame({
    "nb" : tables["transfer_history"].isna().sum(),
    "pct" : (tables["transfer_history"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1101440, 10)
Proportion de valeurs manquantes par variables dans la table transfer_history


,nb,pct
player_id,0,0.00
season_name,2,0.00
from_team_id,0,0.00
from_team_name,0,0.00
to_team_id,0,0.00
to_team_name,0,0.00
transfer_type,0,0.00
value_at_transfer,0,0.00
transfer_fee,0,0.00
transfer_date,580,0.05


In [337]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["transfer_history"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["transfer_history"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["transfer_history"].columns[tables["transfer_history"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  580
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.

In [338]:
print("Doublons stricts :", tables["transfer_history"].duplicated().sum())

Doublons stricts : 123


Est-ce que les frais de tranferts et les valeurs ont des valeurs négatives ?

In [339]:
print(
    tables["transfer_history"][
        ["value_at_transfer", "transfer_fee"]
    ].describe()
)

       value_at_transfer  transfer_fee
count       1.101440e+06  1.101440e+06
mean        3.210798e+05  1.137681e+05
std         1.678342e+06  1.501613e+06
min         0.000000e+00  0.000000e+00
25%         0.000000e+00  0.000000e+00
50%         0.000000e+00  0.000000e+00
75%         2.000000e+05  0.000000e+00
max         1.800000e+08  2.220000e+08


### `PLAYER_MARKET_VALUES`

In [340]:
print(tables["player_market_value"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_market_value")
pd.DataFrame({
    "nb" : tables["player_market_value"].isna().sum(),
    "pct" : (tables["player_market_value"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(901429, 3)
Proportion de valeurs manquantes par variables dans la table player_market_value


,nb,pct
player_id,0,0.0
date_unix,0,0.0
value,0,0.0


In [341]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_market_value"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_market_value"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_market_value"].columns[tables["player_market_value"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  0
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


In [342]:
print("Doublons stricts :", tables["player_market_value"].duplicated().sum())

Doublons stricts : 0


valeurs négatives ?

In [343]:
print(
    tables["player_market_value"][
        ["value"]
    ].describe()
)

              value
count  9.014290e+05
mean   1.544903e+06
std    5.424065e+06
min    0.000000e+00
25%    1.000000e+05
50%    3.000000e+05
75%    8.000000e+05
max    2.000000e+08


### `PLAYER_PERFORMANCES`


In [344]:
print(tables["player_performances"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_performances")
pd.DataFrame({
    "nb" : tables["player_performances"].isna().sum(),
    "pct" : (tables["player_performances"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1878719, 20)
Proportion de valeurs manquantes par variables dans la table player_performances


,nb,pct
player_id,0,0.00
penalty_goals,0,0.00
direct_red_cards,0,0.00
second_yellow_cards,0,0.00
yellow_cards,0,0.00
subed_out,0,0.00
subed_in,0,0.00
own_goals,0,0.00
goals_conceded,0,0.00
assists,0,0.00
